# Derived Data Types

So far, every message was a contiguous array of elements of one basic type.
Real data is often neither contiguous nor homogeneous: a column of a matrix, the boundary layer of a 2D tile, a struct with members of different types.
**Derived data types** describe such data layouts, so that MPI can send them directly — without copying the data by hand.

**After working through this notebook you will be able to**

* explain type maps, type signatures, size and extent of a data type,
* create, commit and free derived types with `MPI_Type_contiguous()`, `MPI_Type_vector()`, `MPI_Type_indexed()`, `MPI_Type_create_struct()` and `MPI_Type_create_subarray()`,
* control the extent of a type with `MPI_Type_create_resized()`, and avoid the typical pitfalls with arrays of types,
* pack and unpack data explicitly with `MPI_Pack()` and `MPI_Unpack()` — for messages whose layout is only known at run time —
  and measure how derived types compare with manual packing.

The notebook [Virtual Topologies and Neighborhood Collectives](topologies.ipynb) builds on this one: its 2D ghost cell exchange sends the rows and columns of a grid with derived types.

As before, prompts marked with 🤔 **Predict** ask you to think about the result *before* running the next cell.

## Working with this Notebook

The C code cells are compiled and run with the `%%mpi` cell magic: a cell is the **body of `main()`**, with `rank` and `size` already set up, unless you pass `-r`.
The options used in this notebook are

| Option | Meaning |
|---|---|
| `-n N` | start `N` MPI processes (default: 4) |
| `-T S` | kill the program after `S` seconds (default: 60) |
| `-i HEADER …` | include additional headers |
| `-O VAR` | store the program output as a string in the Python variable `VAR` |
| `-o FILE` | where to write the source code |

Load the extension once per kernel session:

In [ ]:
%load_ext ice.magic

## Motivation

### Heterogeneous Data

Rank 0 reads the configuration of a simulation from a file and broadcasts it to all other ranks.
With basic data types, every member of the struct needs its own broadcast:

In [ ]:
%%mpi -n 4 -o code/dt-config-members.c
struct config { int nx, ny; double du; int it; };
struct config cfg = { 0, 0, 0.0, 0 };

if (rank == 0) {
    // in a real code: read from a file
    cfg.nx = 100;
    cfg.ny = 200;
    cfg.du = 0.01;
    cfg.it = 500;
}

// one broadcast per member
MPI_Bcast(&cfg.nx, 1, MPI_INT,    0, MPI_COMM_WORLD);
MPI_Bcast(&cfg.ny, 1, MPI_INT,    0, MPI_COMM_WORLD);
MPI_Bcast(&cfg.du, 1, MPI_DOUBLE, 0, MPI_COMM_WORLD);
MPI_Bcast(&cfg.it, 1, MPI_INT,    0, MPI_COMM_WORLD);

printf("rank %d: nx = %d, ny = %d, du = %g, it = %d\n", rank, cfg.nx, cfg.ny, cfg.du, cfg.it);

That's four messages instead of one, and the code has to be kept in sync with the struct.
What we want is `MPI_Bcast(&cfg, 1, <type of struct config>, 0, comm)`.

Sending the struct as raw bytes, `MPI_Bcast(&cfg, sizeof(cfg), MPI_BYTE, 0, comm)`, is **not** a solution: `MPI_BYTE` is never converted, so the program breaks on heterogeneous systems with different byte orders or type sizes, and MPI has no idea what the message contains.

### Non-Contiguous Data

In C, matrices are stored row by row.
A row of a matrix is contiguous in memory, a **column is not**:

![Top: MPI_Type_vector with count 2, blocklength 3 and stride 5 selects the elements 0, 1, 2 and 5, 6, 7. Bottom: a 6 by 5 matrix with column 1 highlighted, and its memory layout: the elements of the column are 5 elements apart](img/vector.svg)

To send a column, you could send every element on its own — many tiny messages, each paying the full latency — or copy the elements into a contiguous buffer by hand and send that.
Derived data types do better.

## Type Maps and Type Signatures

A general MPI data type describes a data layout in memory, so that MPI can *serialize* the data into the message stream on the sender and *deserialize* it on the receiver — networks, like files, are serial channels.
The specification is **declarative**: it says *what* the layout is, and leaves it to the implementation *how* to transfer it efficiently.

Formally, a data type is a sequence of pairs of a basic type and a displacement in bytes, the **type map**:

$$\text{Typemap} = \{(\text{type}_0, \text{disp}_0), \dots, (\text{type}_{n-1}, \text{disp}_{n-1})\}, \qquad \text{e.g. } \{(\texttt{int}, 0), (\texttt{double}, 8), (\texttt{int}, 64)\}$$

The sequence of the basic types alone, with the displacements dropped, is the **type signature**, here $\{\texttt{int}, \texttt{double}, \texttt{int}\}$.
The displacements need not be distinct, positive, or in increasing order.
Overlapping entries are fine for sending, but a receive into a type in which two entries overlap is erroneous.

![Sender and receiver use different type maps with the same type signature int, double, int. The message contains the data in the order of the signature; the receiver places it at its own displacements](img/typemap.svg)

**Type matching** only concerns the signatures: the sender and the receiver may use completely different type maps, as long as their type signatures match.
This gives a handy trick, used throughout this notebook: **send with a derived type and receive as plain contiguous elements** — the receive buffer then shows exactly which elements the type selects, in which order.

## Creating Derived Types

A derived type is created in three steps:

1. **construct** it from existing types with one of the `MPI_Type_*()` constructors,
2. **commit** it with `MPI_Type_commit(MPI_Datatype *newtype)` — only committed types can be used for communication,
3. **free** it with `MPI_Type_free(MPI_Datatype *newtype)` when it is no longer needed.

All of these are **local** calls — no communication, no collective.
Derived types can be built from other derived types; a type that is only used as a building block need not be committed.

## Contiguous

```c
int MPI_Type_contiguous(int count, MPI_Datatype oldtype, MPI_Datatype *newtype);
```

The simplest constructor: `count` copies of `oldtype`, one after the other.

> 🤔 **Predict:** Two "triples" are sent and received as six plain `int`s. What does `b` contain?

In [ ]:
%%mpi -n 1 -o code/dt-contiguous.c
int a[6] = { 0, 1, 2, 3, 4, 5 };
int b[6] = { -1, -1, -1, -1, -1, -1 };

MPI_Datatype triple;
MPI_Type_contiguous(3, MPI_INT, &triple);
MPI_Type_commit(&triple);

// send 2 triples to myself and receive them as 6 ints: the type signatures match
MPI_Sendrecv(a, 2, triple, rank, 0, b, 6, MPI_INT, rank, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

MPI_Type_free(&triple);

printf("b = { %d, %d, %d, %d, %d, %d }\n", b[0], b[1], b[2], b[3], b[4], b[5]);

## Vector

```c
int MPI_Type_vector(int count, int blocklength, int stride, MPI_Datatype oldtype, MPI_Datatype *newtype);
```

`count` blocks of `blocklength` elements each, with the starts of consecutive blocks `stride` elements apart, all in units of `oldtype` — see the figure in the motivation above: `MPI_Type_vector(2, 3, 5, MPI_INT, …)` selects the elements 0, 1, 2 and 5, 6, 7.

In [ ]:
%%mpi -n 1 -o code/dt-vector.c
int a[13], b[6];
for (int i = 0; i < 13; ++i)
    a[i] = i;

MPI_Datatype nt;
MPI_Type_vector(2, 3, 5, MPI_INT, &nt);         // 2 blocks of 3 elements, a new block every 5 elements
MPI_Type_commit(&nt);

MPI_Sendrecv(a, 1, nt, rank, 0, b, 6, MPI_INT, rank, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
MPI_Type_free(&nt);

printf("received:");
for (int i = 0; i < 6; ++i)
    printf(" %d", b[i]);
printf("\n");

A column of a matrix with `nrows` rows and `ncols` columns is a vector of `nrows` blocks of one element, with a stride of `ncols`.
The column is selected by the address passed as buffer: `&m[0][1]` for column 1.
Rank 0 sends the column, and rank 1 receives it as a contiguous array:

In [ ]:
%%mpi -n 2 -o code/dt-column.c
const int nrows = 6, ncols = 5;

MPI_Datatype column;
MPI_Type_vector(nrows, 1, ncols, MPI_INT, &column);
MPI_Type_commit(&column);

if (rank == 0) {
    int m[6][5];
    for (int r = 0; r < nrows; ++r)
        for (int c = 0; c < ncols; ++c)
            m[r][c] = r * ncols + c;

    MPI_Send(&m[0][1], 1, column, 1, 0, MPI_COMM_WORLD);     // column 1
} else if (rank == 1) {
    int col[6];
    MPI_Recv(col, nrows, MPI_INT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

    printf("rank 1: column =");
    for (int r = 0; r < nrows; ++r)
        printf(" %d", col[r]);
    printf("\n");
}

MPI_Type_free(&column);

The receiver could just as well use the column type to receive directly into a column of its own matrix.

## Size and Extent

Two numbers characterize a data type:

* the **size**: the number of bytes of *data* in the type, i.e., in the message — gaps don't count,
* the **extent**: the span from the first to the last byte, rounded up to the alignment of the basic types:

$$\text{lb} = \min_j \text{disp}_j, \qquad \text{ub} = \max_j(\text{disp}_j + \text{sizeof}(\text{type}_j)) + \varepsilon, \qquad \text{extent} = \text{ub} - \text{lb}$$

where $\varepsilon$ is the padding needed for alignment.
The **true extent** is the span without that padding.

![Size and extent: the type map double at 0, char at 8 has a size of 9 bytes and an extent of 16 bytes because of alignment padding. MPI_Type_vector(2, 3, 5, MPI_INT) has a size of 24 bytes and an extent of 32 bytes](img/extent.svg)

```c
int MPI_Type_size(MPI_Datatype datatype, int *size);
int MPI_Type_get_extent(MPI_Datatype datatype, MPI_Aint *lb, MPI_Aint *extent);
int MPI_Type_get_true_extent(MPI_Datatype datatype, MPI_Aint *true_lb, MPI_Aint *true_extent);
```

`MPI_Aint` is the MPI integer type for addresses and byte displacements — large enough for any address.
Since MPI 4.0, these calls also exist with the suffix `_c`, e.g. `MPI_Type_size_c()`, which return `MPI_Count` values for types larger than 2 GiB.

> 🤔 **Predict:** Size and extent of a column of a 6 × 5 `int` matrix?

In [ ]:
%%mpi -n 1 -o code/dt-size-extent.c
MPI_Datatype types[3];
const char *names[3] = { "vector(2, 3, 5, MPI_INT)", "column of a 6 x 5 int matrix", "{(double, 0), (char, 8)}" };

MPI_Type_vector(2, 3, 5, MPI_INT, &types[0]);
MPI_Type_vector(6, 1, 5, MPI_INT, &types[1]);

int          blocklengths[2] = { 1, 1 };
MPI_Aint     displs[2]       = { 0, 8 };
MPI_Datatype oldtypes[2]     = { MPI_DOUBLE, MPI_CHAR };
MPI_Type_create_struct(2, blocklengths, displs, oldtypes, &types[2]);

for (int k = 0; k < 3; ++k) {
    int nbytes;
    MPI_Aint lb, extent, true_lb, true_extent;
    MPI_Type_size(types[k], &nbytes);
    MPI_Type_get_extent(types[k], &lb, &extent);
    MPI_Type_get_true_extent(types[k], &true_lb, &true_extent);
    printf("%-30s size %3d, lb %ld, extent %3ld, true extent %3ld bytes\n",
           names[k], nbytes, (long)lb, (long)extent, (long)true_extent);
    MPI_Type_free(&types[k]);
}

The column has the same size as the vector, 24 bytes, but its extent is ((6 − 1) · 5 + 1) · 4 = 104 bytes: it ends with the last element of the column, not at the end of the last row.
The extent of the struct type is padded to 16 bytes, a multiple of the alignment of `double`; its true extent, 9 bytes, ignores the padding.

### The Extent Determines Where the Next Copy Starts

When a type is used with `count > 1` — or as the old type of another constructor — the copies are placed **one extent apart**.
For most types this is what you want.
For the column type, it isn't:

![Left: count = 2 of the column type; the second copy starts one extent, i.e. 26 ints, after the first: at element 27 in the last row, followed by 32, 37, … outside the matrix. Right: with the extent resized to one int, the second copy is the next column](img/resized.svg)

> 🤔 **Predict:** The cell sends `count = 2` of the column type, starting at column 1. Which elements arrive? (The matrix is followed by memory filled with -1.)

In [ ]:
%%mpi -n 1 -o code/dt-resized.c
const int nrows = 6, ncols = 5;

// the matrix, followed by as many elements of -1 - room for reading beyond the matrix
int *m = malloc(2 * nrows * ncols * sizeof(int));
for (int i = 0; i < 2 * nrows * ncols; ++i)
    m[i] = i < nrows * ncols ? i : -1;
int b[12];

MPI_Datatype column, column1;
MPI_Type_vector(nrows, 1, ncols, MPI_INT, &column);
MPI_Type_commit(&column);

// count = 2: the 2nd copy starts one extent (26 ints) after the 1st
MPI_Sendrecv(&m[1], 2, column, rank, 0, b, 12, MPI_INT, rank, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
printf("column, count = 2:            ");
for (int i = 0; i < 12; ++i)
    printf(" %2d", b[i]);
printf("\n");

// the same type with an extent of one int: the 2nd copy is the next column
MPI_Type_create_resized(column, 0, sizeof(int), &column1);
MPI_Type_commit(&column1);
MPI_Sendrecv(&m[1], 2, column1, rank, 0, b, 12, MPI_INT, rank, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
printf("resized column, count = 2:    ");
for (int i = 0; i < 12; ++i)
    printf(" %2d", b[i]);
printf("\n");

MPI_Type_free(&column);
MPI_Type_free(&column1);
free(m);

```c
int MPI_Type_create_resized(MPI_Datatype oldtype, MPI_Aint lb, MPI_Aint extent, MPI_Datatype *newtype);
```

returns a type with the same type map, but with the given lower bound and extent.
It changes neither the size nor the content of a message — only where *consecutive* copies of the type start.
(Older material uses the markers `MPI_LB` and `MPI_UB` for this; they were removed in MPI 3.0.)

The resized column type is the standard tool for scattering, gathering or transposing matrices by columns — see Exercise 2.

## Indexed

```c
int MPI_Type_indexed(int count, const int blocklengths[], const int displacements[],
                     MPI_Datatype oldtype, MPI_Datatype *newtype);
```

| Argument | Meaning |
|---|---|
| `count` | number of blocks — the number of entries in both arrays |
| `blocklengths[i]` | number of elements in block `i` |
| `displacements[i]` | start of block `i`, in units of the **extent** of `oldtype` |

An indexed type generalizes the vector: the blocks may have different lengths and arbitrary positions.

The following example has an old type with two `double`s, i.e., an extent of 16 bytes; block 0 holds 3 copies starting at displacement 4 (64 bytes), block 1 one copy at displacement 0.

> 🤔 **Predict:** Size and extent of the new type? Which elements of `a` does it select, in which order?

In [ ]:
%%mpi -n 1 -o code/dt-indexed.c
double a[14], b[8];
for (int i = 0; i < 14; ++i)
    a[i] = i;

MPI_Datatype pair, nt;
MPI_Type_contiguous(2, MPI_DOUBLE, &pair);                      // {(double, 0), (double, 8)}, extent 16

int blocklengths[2]  = { 3, 1 };
int displacements[2] = { 4, 0 };                                // in units of the extent of pair
MPI_Type_indexed(2, blocklengths, displacements, pair, &nt);
MPI_Type_commit(&nt);

int nbytes;
MPI_Aint lb, extent;
MPI_Type_size(nt, &nbytes);
MPI_Type_get_extent(nt, &lb, &extent);
printf("size %d bytes, lb %ld, extent %ld bytes\n", nbytes, (long)lb, (long)extent);

MPI_Sendrecv(a, 1, nt, rank, 0, b, 8, MPI_DOUBLE, rank, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
printf("selected:");
for (int i = 0; i < 8; ++i)
    printf(" %g", b[i]);
printf("\n");

MPI_Type_free(&pair);
MPI_Type_free(&nt);

The type map is $\{(\texttt{double}, 64), (\texttt{double}, 72), \dots, (\texttt{double}, 104), (\texttt{double}, 0), (\texttt{double}, 8)\}$: the data is sent in the order of the type map, not in the order of the addresses.
The extent spans from byte 0 to byte 112 — not 16, which is the extent of the old type.

A more realistic example: the **lower triangle** of a matrix.
Row `i` contributes a block of `i + 1` elements, starting at element `i · N`:

![The lower triangle of a 6 by 6 matrix: row i is a block of i + 1 elements starting at element 6 i](img/indexed.svg)

With the same type on both sides, `MPI_Sendrecv()` copies the lower triangle of `a` into `b` — here on the same process:

In [ ]:
%%mpi -n 1 -o code/dt-lower-triangle.c
#define N 6

double a[N * N], b[N * N];
for (int i = 0; i < N * N; ++i) {
    a[i] = i;
    b[i] = 0;
}

int lens[N], displs[N];
for (int i = 0; i < N; ++i) {
    displs[i] = i * N;
    lens[i]   = i + 1;
}

MPI_Datatype tril;
MPI_Type_indexed(N, lens, displs, MPI_DOUBLE, &tril);
MPI_Type_commit(&tril);

MPI_Sendrecv(a, 1, tril, rank, 0, b, 1, tril, rank, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

MPI_Type_free(&tril);

for (int r = 0; r < N; ++r) {
    for (int c = 0; c < N; ++c)
        printf(" %3.0f", b[r * N + c]);
    printf("\n");
}

If all blocks have the same length, `MPI_Type_create_indexed_block(count, blocklength, displacements, oldtype, &newtype)` saves the array of block lengths (Exercise 1).

## Displacements in Bytes and Addresses

The constructors with an `h` take the displacements and strides **in bytes** instead of in units of the old type:
`MPI_Type_create_hvector()`, `MPI_Type_create_hindexed()`, `MPI_Type_create_hindexed_block()` — and `MPI_Type_create_struct()` below.
Byte displacements have the type `MPI_Aint`.

**Use the MPI routines for address computations** — pointer arithmetic is not portable for this purpose:

```c
int MPI_Get_address(const void *location, MPI_Aint *address);
MPI_Aint MPI_Aint_diff(MPI_Aint addr1, MPI_Aint addr2);       // addr1 - addr2
MPI_Aint MPI_Aint_add(MPI_Aint base, MPI_Aint disp);          // base + disp
```

Usually, displacements are relative to the buffer address passed to the communication call.
Absolute addresses are allowed as well; the buffer address is then `MPI_BOTTOM`.

In [ ]:
%%mpi -n 1 -o code/dt-addresses.c
double a[100];
MPI_Aint a1, a2;

MPI_Get_address(&a[0],  &a1);
MPI_Get_address(&a[50], &a2);

printf("displacement of a[50] relative to a[0]: %ld bytes\n", (long)MPI_Aint_diff(a2, a1));

## Struct

```c
int MPI_Type_create_struct(int count, const int blocklengths[], const MPI_Aint displacements[],
                           const MPI_Datatype types[], MPI_Datatype *newtype);
```

The most general constructor: every block has its own length, byte displacement **and type**.
This is what the configuration struct from the motivation needs:

![The memory layout of struct config: nx and ny at bytes 0 and 4, du at byte 8, it at byte 16, followed by 4 bytes of padding. Block 0 is 2 MPI_INT at displacement 0, block 1 one MPI_DOUBLE at 8, block 2 one MPI_INT at 16](img/struct.svg)

The displacements are obtained with `MPI_Get_address()` relative to the start of the struct — the compiler decides about the padding, so never compute them by hand.
(`offsetof()` from `<stddef.h>` gives the same values.)

In [ ]:
%%mpi -n 4 -o code/dt-config-struct.c
struct config { int nx, ny; double du; int it; };
struct config cfg = { 0, 0, 0.0, 0 };

if (rank == 0) {
    cfg.nx = 100;
    cfg.ny = 200;
    cfg.du = 0.01;
    cfg.it = 500;
}

// describe the struct: nx and ny are adjacent, so they form one block
int          blocklengths[3] = { 2, 1, 1 };
MPI_Datatype types[3]        = { MPI_INT, MPI_DOUBLE, MPI_INT };
MPI_Aint     displs[3], base;

MPI_Get_address(&cfg,    &base);
MPI_Get_address(&cfg.nx, &displs[0]);
MPI_Get_address(&cfg.du, &displs[1]);
MPI_Get_address(&cfg.it, &displs[2]);
for (int i = 0; i < 3; ++i)
    displs[i] = MPI_Aint_diff(displs[i], base);         // relative to the start of the struct

MPI_Datatype cfgtype;
MPI_Type_create_struct(3, blocklengths, displs, types, &cfgtype);
MPI_Type_commit(&cfgtype);

MPI_Bcast(&cfg, 1, cfgtype, 0, MPI_COMM_WORLD);          // the whole struct in one call

if (rank == 0) {
    MPI_Aint lb, extent;
    MPI_Type_get_extent(cfgtype, &lb, &extent);
    printf("extent of cfgtype: %ld bytes, sizeof(struct config): %zu bytes\n", (long)extent, sizeof(struct config));
}
printf("rank %d: nx = %d, ny = %d, du = %g, it = %d\n", rank, cfg.nx, cfg.ny, cfg.du, cfg.it);

MPI_Type_free(&cfgtype);

### Arrays of Structs

For a whole struct, the extent of the MPI type matches `sizeof` — the alignment rule $\varepsilon$ reproduces the padding of the compiler in common cases.
Often, however, only **some members** are sent — e.g., only the positions of an array of particles.
The extent of such a type covers only these members, so with `count > 1` the second copy does not start at the second particle.
Again, `MPI_Type_create_resized()` sets the extent to `sizeof` the struct.

> 🤔 **Predict:** What does rank 1 receive if you send with `pos` instead of `pos_resized`? Try it.

In [ ]:
%%mpi -n 2 -o code/dt-struct-array.c
struct particle { double x[3]; double v[3]; double m; int id; };
#define NP 4

// the position: 3 doubles at the start of the struct ...
MPI_Datatype pos, pos_resized;
MPI_Type_contiguous(3, MPI_DOUBLE, &pos);
MPI_Type_commit(&pos);
// ... with the extent of a whole particle, so that consecutive copies are consecutive particles
MPI_Type_create_resized(pos, 0, sizeof(struct particle), &pos_resized);
MPI_Type_commit(&pos_resized);

if (rank == 0) {
    struct particle p[NP] = { 0 };                              // zeroed, including the padding
    for (int k = 0; k < NP; ++k) {
        for (int d = 0; d < 3; ++d) {
            p[k].x[d] = 10 * k + d;
            p[k].v[d] = -1.0;
        }
        p[k].m  = -1.0;
        p[k].id = k;
    }
    MPI_Send(p, NP, pos_resized, 1, 0, MPI_COMM_WORLD);         // the positions of all particles
} else if (rank == 1) {
    double x[NP][3];
    MPI_Recv(x, 3 * NP, MPI_DOUBLE, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    for (int k = 0; k < NP; ++k)
        printf("rank 1: particle %d at (%4.1f, %4.1f, %4.1f)\n", k, x[k][0], x[k][1], x[k][2]);
}

MPI_Type_free(&pos);
MPI_Type_free(&pos_resized);

With `pos`, consecutive copies are only 24 bytes apart — the extent of 3 `double`s — instead of `sizeof(struct particle)` = 64 bytes.
Rank 1 receives the position of particle 0, then its velocity (-1, -1, -1), then its mass, its `id` with the padding read as a `double`, and the first coordinate of particle 1, and so on.
The type signatures match, so MPI reports no error — only the extent tells MPI where the next particle starts.

## Subarray

```c
int MPI_Type_create_subarray(int ndims, const int sizes[], const int subsizes[], const int starts[],
                             int order, MPI_Datatype oldtype, MPI_Datatype *newtype);
```

| Argument | Meaning |
|---|---|
| `ndims` | number of dimensions of the array |
| `sizes[]` | size of the full array in each dimension |
| `subsizes[]` | size of the subarray in each dimension |
| `starts[]` | start of the subarray in each dimension — counted from 0, also in Fortran |
| `order` | `MPI_ORDER_C` (row-major) or `MPI_ORDER_FORTRAN` (column-major) |

A subarray type selects a rectangular block of a multi-dimensional array — e.g., the *bulk* of a matrix without its boundary.

![A 6 by 5 matrix with the inner 4 by 3 block highlighted. With MPI_ORDER_C, sizes is {6, 5}, i.e. rows first, subsizes {4, 3}, starts {1, 1}](img/subarray.svg)

With `MPI_ORDER_C`, the dimensions are given **from the slowest to the fastest** — for a matrix, the number of rows first.
A common mistake is `{ncols, nrows}`, which is only correct with `MPI_ORDER_FORTRAN`.

Unlike for the vector type, the buffer address is the start of the **full** array, and the extent of the type is the extent of the full array.

In [ ]:
%%mpi -n 1 -o code/dt-subarray.c
const int nrows = 6, ncols = 5;

int m[6 * 5], b[12];
for (int i = 0; i < nrows * ncols; ++i)
    m[i] = i;

int sizes[2]    = { nrows, ncols };             // C order: the slowest dimension first
int subsizes[2] = { nrows - 2, ncols - 2 };
int starts[2]   = { 1, 1 };

MPI_Datatype bulk;
MPI_Type_create_subarray(2, sizes, subsizes, starts, MPI_ORDER_C, MPI_INT, &bulk);
MPI_Type_commit(&bulk);

MPI_Aint lb, extent;
MPI_Type_get_extent(bulk, &lb, &extent);
printf("extent of the subarray type: %ld bytes\n", (long)extent);

// the buffer is the start of the full array
MPI_Sendrecv(m, 1, bulk, rank, 0, b, 12, MPI_INT, rank, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
MPI_Type_free(&bulk);

for (int r = 0; r < nrows - 2; ++r) {
    for (int c = 0; c < ncols - 2; ++c)
        printf(" %2d", b[r * (ncols - 2) + c]);
    printf("\n");
}

## Packing and Unpacking

Derived types *describe* a layout, and MPI gathers the data during the send.
**Packing** is the explicit alternative: the program gathers the data itself into a contiguous buffer with `MPI_Pack()`, sends the buffer as one message of type `MPI_PACKED`, and the receiver scatters the data to its destinations with `MPI_Unpack()`.

![Rank 0 packs the int j and column 2 of a 6 x 5 matrix into a 28-byte buffer; the position advances from 0 to 4 to 28. The buffer is sent as MPI_PACKED, and rank 1 unpacks it into j and a contiguous array col](img/pack.svg)

Why copy the data by hand when a derived type does it for you?
A derived type must be known **before** the communication, on both sides — including every count in its type signature.
Packing is the tool when that is not the case:

* **the type signature depends on the data**: a count followed by that many values, a list of records of different lengths, a sparse vector.
  Sender and receiver still agree on the *format* in advance — e.g. "an `int` `nnz`, then `nnz` indices, then `nnz` values" — but the format can refer to values in the message itself: the receiver unpacks `nnz` first and uses it as the count of the next `MPI_Unpack()`,
* **the message is assembled piece by piece**: in a loop, or by different parts of a program or library, each with its own type — without constructing a struct type for every combination,
* **many small messages are bundled into one**, which saves the latency of all but one of them,
* **the data must be portable**, e.g. stored in a file or read by a different MPI library — this needs the variant `MPI_Pack_external()`, see `external32` below; the format of `MPI_Pack()` itself is implementation-specific.

For data with a fixed, regular layout, derived types are usually the better choice: they need no extra buffer, and they allow zero-copy communication (see [Performance](#Performance)).

### The Interface

```c
int MPI_Pack(const void *inbuf, int incount, MPI_Datatype datatype,
             void *outbuf, int outsize, int *position, MPI_Comm comm);
int MPI_Unpack(const void *inbuf, int insize, int *position,
               void *outbuf, int outcount, MPI_Datatype datatype, MPI_Comm comm);
int MPI_Pack_size(int incount, MPI_Datatype datatype, MPI_Comm comm, int *size);
```

The two calls are mirror images of each other:

| | `MPI_Pack()` | `MPI_Unpack()` |
|---|---|---|
| user data | `inbuf`, `incount`, `datatype`: the data to pack — like the buffer of a send | `outbuf`, `outcount`, `datatype`: where to put the data — like the buffer of a receive |
| pack buffer | `outbuf`, `outsize`: the buffer and its **total** size in bytes | `inbuf`, `insize`: the buffer and its **total** size in bytes |
| `position` | in: where to start writing, in bytes; out: the first byte after the packed data | in: where to start reading, in bytes; out: the first byte after the unpacked data |
| `comm` | the communicator the buffer will be sent on | the communicator it was received on |

* **`position` is a cursor.** Set it to 0 before the first call; every call advances it by the number of bytes written or read.
  After packing, `position` is exactly the number of bytes to send.
  `outsize` and `insize` are the sizes of the whole buffer, not the remaining space — MPI computes that from `position`; packing beyond the end of the buffer is an error.
* The user data can be described by **any** data type, including derived types: a single `MPI_Pack()` gathers a whole matrix column, and `MPI_Unpack()` can scatter into one.
* **Why a communicator?** The packed representation may depend on the processes that receive it: on a heterogeneous system, MPI may have to convert the data (byte order, sizes of the types) or prepend a header that describes the representation.
  On a homogeneous cluster, the packed data typically is just the raw bytes.
* **`MPI_Pack_size()`** returns an **upper bound** for the number of bytes that packing `incount` elements of `datatype` needs — it may be larger than the data, e.g. to leave room for a header.
  For a buffer that is filled by several calls, add up the bounds of the individual calls.
* The packed buffer is sent and received with the type **`MPI_PACKED`**, with a count in **bytes**; `MPI_Get_count(&status, MPI_PACKED, &nbytes)` returns the number of bytes received.
* The pack buffer is **opaque**: never read values from it directly, e.g. by casting `buffer + 4` to a `double *` — the format is implementation-specific, and the values are generally not aligned. Always use `MPI_Unpack()`.

### The Position Cursor

This is the example of the figure: rank 0 packs the index `j` of a column and the column itself — with the vector type from above — into one buffer.
Rank 1 does not need a derived type: it unpacks the index, and then the column into a contiguous array.

> 🤔 **Predict:** what is the value of `position` after each `MPI_Pack()`? How many bytes does rank 1 receive?

In [ ]:
%%mpi -n 2 -o code/dt-pack-column.c
const int nrows = 6, ncols = 5;
char buffer[256];                       // the pack buffer: large enough for this example
int position = 0;                       // the cursor: the next free byte in the buffer

if (rank == 0) {
    int m[6 * 5];
    for (int i = 0; i < nrows * ncols; ++i)
        m[i] = i;

    MPI_Datatype column;
    MPI_Type_vector(nrows, 1, ncols, MPI_INT, &column);
    MPI_Type_commit(&column);

    int j = 2;                          // which column to send
    printf("rank 0: position %3d at the start\n", position);
    MPI_Pack(&j, 1, MPI_INT, buffer, sizeof buffer, &position, MPI_COMM_WORLD);
    printf("rank 0: position %3d after packing the column index\n", position);
    MPI_Pack(&m[j], 1, column, buffer, sizeof buffer, &position, MPI_COMM_WORLD);
    printf("rank 0: position %3d after packing the column\n", position);

    // send exactly the packed bytes
    MPI_Send(buffer, position, MPI_PACKED, 1, 0, MPI_COMM_WORLD);
    MPI_Type_free(&column);
} else if (rank == 1) {
    MPI_Status status;
    int nbytes;
    MPI_Recv(buffer, sizeof buffer, MPI_PACKED, 0, 0, MPI_COMM_WORLD, &status);
    MPI_Get_count(&status, MPI_PACKED, &nbytes);

    // unpack in the same order: the column index, then the column - into a contiguous array
    int j, col[6];
    MPI_Unpack(buffer, nbytes, &position, &j, 1, MPI_INT, MPI_COMM_WORLD);
    MPI_Unpack(buffer, nbytes, &position, col, nrows, MPI_INT, MPI_COMM_WORLD);

    printf("rank 1: received %d bytes, column %d:", nbytes, j);
    for (int i = 0; i < nrows; ++i)
        printf(" %2d", col[i]);
    printf("\n");
}

The packed data is just the raw data: 4 bytes for `j`, and 6 × 4 bytes for the column — no header, no padding.
That is typical for a homogeneous system, but it is not guaranteed, so always use `position` and `MPI_Get_count()` instead of computing sizes yourself.

Rank 1 receives into a buffer of 256 bytes, larger than the message; it passes the actual length `nbytes` as `insize` to `MPI_Unpack()`.
Packing and unpacking use the same type signature — one `int`, then six `int`s — but different type maps: the column is gathered from a strided layout and scattered into a contiguous one.

### A Layout Known Only While Unpacking

A common case: the sender sends a count, followed by that many values.
The receiver does not know the count in advance, so it can neither construct a matching derived type nor allocate a receive buffer of the right size.
With packing, it first probes the message for its size in bytes, receives the packed buffer, unpacks the count, allocates the array, and then unpacks the values:

In [ ]:
%%mpi -n 2 -o code/dt-pack.c
int position = 0;

if (rank == 0) {
    int n = 5;
    double values[5] = { 1.5, 2.5, 3.5, 4.5, 5.5 };

    // an upper bound for the packed size of the count and the values
    int size_n, size_values;
    MPI_Pack_size(1, MPI_INT,    MPI_COMM_WORLD, &size_n);
    MPI_Pack_size(n, MPI_DOUBLE, MPI_COMM_WORLD, &size_values);
    const int bufsize = size_n + size_values;
    char *buffer = malloc(bufsize);

    MPI_Pack(&n, 1, MPI_INT, buffer, bufsize, &position, MPI_COMM_WORLD);
    MPI_Pack(values, n, MPI_DOUBLE, buffer, bufsize, &position, MPI_COMM_WORLD);
    MPI_Send(buffer, position, MPI_PACKED, 1, 0, MPI_COMM_WORLD);
    printf("rank 0: sent %d packed bytes\n", position);
    free(buffer);
} else if (rank == 1) {
    // the receiver does not know the size of the message: probe it first
    MPI_Status status;
    int nbytes;
    MPI_Probe(0, 0, MPI_COMM_WORLD, &status);
    MPI_Get_count(&status, MPI_PACKED, &nbytes);
    char *buffer = malloc(nbytes);
    MPI_Recv(buffer, nbytes, MPI_PACKED, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

    // first the count, then as many values
    int n;
    MPI_Unpack(buffer, nbytes, &position, &n, 1, MPI_INT, MPI_COMM_WORLD);
    double *values = malloc(n * sizeof(double));
    MPI_Unpack(buffer, nbytes, &position, values, n, MPI_DOUBLE, MPI_COMM_WORLD);

    printf("rank 1: %d values:", n);
    for (int i = 0; i < n; ++i)
        printf(" %g", values[i]);
    printf("\n");
    free(values);
    free(buffer);
}

Without packing, the count and the values would need two messages — and the latency of two messages.
(For this particular case, there is also a third option: send only the values, and let the receiver compute the count with `MPI_Probe()` and `MPI_Get_count(&status, MPI_DOUBLE, &n)`.
That no longer works as soon as the message contains more than one variable-length piece.)

### Records of Variable Length

Packing shines when a message consists of several pieces whose number and lengths vary.
Rank 0 sends a list of named fields: each record is the length of the name, the name, the number of values, and the values.
The receiver knows the format of a record, but neither the number of records nor their lengths: it reads each length before the data it describes, and unpacks record after record **until the cursor reaches the end of the message**, `position == nbytes`.

> 🤔 **Predict:** how many bytes does the message have? At which byte offset do the values of `temperature` start — would it be safe to read them through a `double *` pointing into the buffer?

In [ ]:
%%mpi -n 2 -i string.h -o code/dt-pack-records.c
// a record: a name, then a number of values; the records differ in both
struct record { const char *name; int n; double values[4]; };

if (rank == 0) {
    struct record rec[3] = {
        { "pressure",    1, { 101.3 } },
        { "velocity",    3, { 1.0, -0.5, 0.25 } },
        { "temperature", 2, { 293.15, 295.0 } },
    };

    // an upper bound for the packed size, record by record
    int bufsize = 0;
    for (int r = 0; r < 3; ++r) {
        int s;
        int len = strlen(rec[r].name);
        MPI_Pack_size(1, MPI_INT, MPI_COMM_WORLD, &s);              // len and n: two MPI_Pack() calls
        bufsize += 2 * s;
        MPI_Pack_size(len, MPI_CHAR, MPI_COMM_WORLD, &s);           // the name
        bufsize += s;
        MPI_Pack_size(rec[r].n, MPI_DOUBLE, MPI_COMM_WORLD, &s);    // the values
        bufsize += s;
    }
    char *buffer = malloc(bufsize);

    int position = 0;
    for (int r = 0; r < 3; ++r) {
        int len = strlen(rec[r].name);
        MPI_Pack(&len,          1,        MPI_INT,    buffer, bufsize, &position, MPI_COMM_WORLD);
        MPI_Pack(rec[r].name,   len,      MPI_CHAR,   buffer, bufsize, &position, MPI_COMM_WORLD);
        MPI_Pack(&rec[r].n,     1,        MPI_INT,    buffer, bufsize, &position, MPI_COMM_WORLD);
        MPI_Pack(rec[r].values, rec[r].n, MPI_DOUBLE, buffer, bufsize, &position, MPI_COMM_WORLD);
    }
    printf("rank 0: packed 3 records into %d of %d bytes\n", position, bufsize);
    MPI_Send(buffer, position, MPI_PACKED, 1, 0, MPI_COMM_WORLD);
    free(buffer);
} else if (rank == 1) {
    MPI_Status status;
    int nbytes;
    MPI_Probe(0, 0, MPI_COMM_WORLD, &status);
    MPI_Get_count(&status, MPI_PACKED, &nbytes);
    char *buffer = malloc(nbytes);
    MPI_Recv(buffer, nbytes, MPI_PACKED, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

    // the receiver neither knows how many records there are nor how long they are:
    // it unpacks until the cursor reaches the end of the message
    int position = 0;
    while (position < nbytes) {
        // every length is unpacked before the data it describes: allocate just in time
        int len, n;
        MPI_Unpack(buffer, nbytes, &position, &len,  1,   MPI_INT,    MPI_COMM_WORLD);
        char *name = malloc(len + 1);
        MPI_Unpack(buffer, nbytes, &position, name,  len, MPI_CHAR,   MPI_COMM_WORLD);
        name[len] = '\0';
        MPI_Unpack(buffer, nbytes, &position, &n,    1,   MPI_INT,    MPI_COMM_WORLD);
        double *values = malloc(n * sizeof(double));
        MPI_Unpack(buffer, nbytes, &position, values, n,  MPI_DOUBLE, MPI_COMM_WORLD);

        printf("rank 1: %-12s", name);
        for (int i = 0; i < n; ++i)
            printf(" %8g", values[i]);
        printf("   (position %d)\n", position);
        free(name);
        free(values);
    }
    free(buffer);
}

The records take 4 + 8 + 4 + 8 = 24, 4 + 8 + 4 + 24 = 40, and 4 + 11 + 4 + 16 = 35 bytes — 99 bytes in total.
The values of `temperature` start at byte 64 + 4 + 11 + 4 = 83, which is not a multiple of 8: a `double *` into the buffer would be misaligned, which is why the buffer must only be read with `MPI_Unpack()`.

On a homogeneous system, `MPI_Pack_size()` typically returns exactly the size of the data — compare the two numbers rank 0 prints — but a portable program must not rely on that: allocate the upper bound, and send `position` bytes.
Note that the bound refers to **one** `MPI_Pack()` call with the same arguments: `len` and `n` are packed by two calls, so the bound for one `int` is counted twice.

### Rules for Packed Messages

* **Unpack what you packed.** The unpack calls must read the same type signature, in the same order, that the pack calls wrote.
  The receiver must therefore know the format of the message; packing only allows it to take counts (or choices, such as the kind of the next record) from the data unpacked so far.
  MPI does not check this: unpacking an `int` where a `double` was packed silently produces garbage.
* **A packing unit is unpacked as a whole.** The data packed into one buffer by a sequence of `MPI_Pack()` calls forms a *packing unit*.
  It must be unpacked by a sequence of `MPI_Unpack()` calls on the whole unit, starting at position 0.
  You can neither concatenate two packed buffers and unpack them as one, nor unpack a piece cut out of a packed buffer on its own — the implementation may have put a header at the start of each unit.
* **Any message can be received as `MPI_PACKED`.** Whatever type the sender used, the receiver may receive the message as `MPI_PACKED` and unpack it piece by piece — e.g., a `struct config` sent with `MPI_Send()` and the type `cfgtype` from above can be received as `MPI_PACKED` and unpacked as two `int`s, a `double` and an `int`.
* **`MPI_PACKED` is not `MPI_BYTE`.** Copying data with `memcpy()` into a `char` buffer and sending it as `MPI_BYTE` works on a homogeneous system, but `MPI_BYTE` data is never converted, while packed data is converted if sender and receiver differ.
  Moreover, `MPI_Pack()` gathers non-contiguous data described by a derived type in a single call.
* **Large buffers:** the `int` arguments limit a pack buffer to 2 GiB. MPI 4.0 added `MPI_Pack_c()`, `MPI_Unpack_c()` and `MPI_Pack_size_c()` with `MPI_Count` arguments.

### Portable Data: `external32`

The format of `MPI_Pack()` is implementation-specific: only the same MPI implementation can unpack it.
For data that leaves the MPI program — stored in a file, or read by a program built with a different MPI library — MPI defines the portable representation **`external32`**: big-endian byte order, IEEE floating point, and fixed sizes (4 bytes for an `int`, 8 for a `double`).
It is the same representation that MPI-IO uses for the data representation `"external32"` (see the MPI-IO notebook).

```c
int MPI_Pack_external(const char datarep[], const void *inbuf, int incount, MPI_Datatype datatype,
                      void *outbuf, MPI_Aint outsize, MPI_Aint *position);
int MPI_Unpack_external(const char datarep[], const void *inbuf, MPI_Aint insize, MPI_Aint *position,
                        void *outbuf, int outcount, MPI_Datatype datatype);
int MPI_Pack_external_size(const char datarep[], int incount, MPI_Datatype datatype, MPI_Aint *size);
```

`datarep` is `"external32"`; there is no communicator, because the representation does not depend on the receivers, and sizes and positions are `MPI_Aint`.

> 🤔 **Predict:** the `int` `0x01020304` and the `double` `1.0` (`0x3FF0000000000000`) are packed in both representations. In which order do you expect the bytes on this machine?

In [ ]:
%%mpi -n 1 -o code/dt-pack-external.c
int    i = 0x01020304;
double d = 1.0;
unsigned char native[64], external[64];
int position = 0;
MPI_Aint xposition = 0;

// the same data, packed in the native and in the portable external32 representation
MPI_Pack(&i, 1, MPI_INT,    native, sizeof native, &position, MPI_COMM_WORLD);
MPI_Pack(&d, 1, MPI_DOUBLE, native, sizeof native, &position, MPI_COMM_WORLD);
MPI_Pack_external("external32", &i, 1, MPI_INT,    external, sizeof external, &xposition);
MPI_Pack_external("external32", &d, 1, MPI_DOUBLE, external, sizeof external, &xposition);

printf("native     (%2d bytes):", position);
for (int k = 0; k < position; ++k)
    printf(" %02x", native[k]);
printf("\nexternal32 (%2ld bytes):", (long)xposition);
for (int k = 0; k < xposition; ++k)
    printf(" %02x", external[k]);
printf("\n");

x86 and ARM processors are little-endian: the native packed data starts with the least significant byte, `04` and `00`.
`external32` always stores the most significant byte first, so the same values are byte-reversed.
On a little-endian machine, `MPI_Pack_external()` therefore has to swap the bytes of every value — portability is not free.

### Derived Types or Packing?

| | Derived types | `MPI_Pack()` / `MPI_Unpack()` |
|---|---|---|
| type signature | must be known completely before the communication, on both sides | follows a format agreed in advance; counts can be taken from the message while unpacking |
| extra memory and copies | none — zero-copy is possible | a pack buffer on both sides, and a copy into and out of it |
| setup | construct and commit a type once, reuse it in every iteration | none, but one call per piece, for every message |
| mixed types (struct-like data) | a struct type with byte displacements | one call per piece, in any order |

**Rule of thumb:** use derived types for fixed, regular layouts that are communicated repeatedly — matrix columns, ghost layers, structs.
Use packing for irregular, data-dependent messages.
The next section measures both for a matrix column.

## Performance

Derived types make **zero-copy** communication possible: the data goes from the user's memory directly to the network, without an intermediate buffer — if the network supports it (RDMA, I/O vectors, shared memory).
Otherwise, the MPI implementation packs the data internally.

* MPI implementations copy internally anyway, e.g. in the network stack or to pack derived types.
* Applications often copy too much: manual packing, unpacking and rearranging.
* Derived types can avoid both — but **some implementations are still slower than manual packing**.
* A simple performance model: the more irregular the type — contiguous, vector, indexed, struct — the more work for the implementation, and the slower it tends to be.

Measure it.
The following benchmark sends a column of an `n × n` matrix of `double`s back and forth between two processes (*ping-pong*) and reports the time per message for

* a **row** — the contiguous baseline with the same amount of data,
* the column **element-wise** — `n` messages of one `double`,
* **manual packing** — copy into a buffer, send, copy out of the buffer on the receiver,
* **`MPI_Type_vector`** — on both sides,
* **`MPI_Pack()`** — with the vector type, sent as `MPI_PACKED`.

The output is stored in `colbench` for plotting.

In [ ]:
%%mpi -n 2 -T 300 -o code/dt-column-bench.c -O colbench
if (size != 2) {
    if (rank == 0)
        fprintf(stderr, "This benchmark requires exactly 2 processes, got %d\n", size);
    MPI_Abort(MPI_COMM_WORLD, 1);
}

const int maxn = argc > 1 ? atoi(argv[1]) : 2048;      // largest matrix dimension
const int nmethods = 5;
const char *names[] = { "row", "element-wise", "manual packing", "MPI_Type_vector", "MPI_Pack" };
const int other = 1 - rank;

double *a   = calloc((size_t)maxn * maxn, sizeof(double)); // the matrix, n x n with n <= maxn
double *buf = malloc(maxn * sizeof(double));

int packmax;
MPI_Datatype col;
MPI_Type_vector(maxn, 1, maxn, MPI_DOUBLE, &col);
MPI_Type_commit(&col);
MPI_Pack_size(1, col, MPI_COMM_WORLD, &packmax);
MPI_Type_free(&col);
char *packbuf = malloc(packmax);

if (rank == 0) {
    printf("%8s", "n");
    for (int k = 0; k < nmethods; ++k)
        printf(" %16s", names[k]);
    printf("\n");
}

for (int n = 16; n <= maxn; n *= 2) {
    MPI_Datatype column;
    MPI_Type_vector(n, 1, n, MPI_DOUBLE, &column);      // column 0 of the n x n matrix
    MPI_Type_commit(&column);
    int reps = n <= 256 ? 200 : 50;

    if (rank == 0)
        printf("%8d", n);

    for (int k = 0; k < nmethods; ++k) {
        double t0 = 0.0;
        for (int r = -reps / 10 - 1; r < reps; ++r) {      // negative r: warm-up, not timed
            if (r == 0) {
                MPI_Barrier(MPI_COMM_WORLD);
                t0 = MPI_Wtime();
            }
            // ping-pong: rank 0 sends first, then receives; rank 1 the other way round
            for (int turn = 0; turn < 2; ++turn) {
                const int sending = (turn == 0) == (rank == 0);
                int pos = 0;
                switch (k) {
                case 0:     // a row: contiguous
                    if (sending) MPI_Send(a, n, MPI_DOUBLE, other, 0, MPI_COMM_WORLD);
                    else         MPI_Recv(a, n, MPI_DOUBLE, other, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
                    break;
                case 1:     // element by element
                    for (int i = 0; i < n; ++i) {
                        if (sending) MPI_Send(&a[(size_t)i * n], 1, MPI_DOUBLE, other, 0, MPI_COMM_WORLD);
                        else         MPI_Recv(&a[(size_t)i * n], 1, MPI_DOUBLE, other, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
                    }
                    break;
                case 2:     // manual packing and unpacking
                    if (sending) {
                        for (int i = 0; i < n; ++i)
                            buf[i] = a[(size_t)i * n];
                        MPI_Send(buf, n, MPI_DOUBLE, other, 0, MPI_COMM_WORLD);
                    } else {
                        MPI_Recv(buf, n, MPI_DOUBLE, other, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
                        for (int i = 0; i < n; ++i)
                            a[(size_t)i * n] = buf[i];
                    }
                    break;
                case 3:     // derived type
                    if (sending) MPI_Send(a, 1, column, other, 0, MPI_COMM_WORLD);
                    else         MPI_Recv(a, 1, column, other, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
                    break;
                case 4:     // MPI_Pack with the derived type
                    if (sending) {
                        MPI_Pack(a, 1, column, packbuf, packmax, &pos, MPI_COMM_WORLD);
                        MPI_Send(packbuf, pos, MPI_PACKED, other, 0, MPI_COMM_WORLD);
                    } else {
                        MPI_Recv(packbuf, packmax, MPI_PACKED, other, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
                        MPI_Unpack(packbuf, packmax, &pos, a, 1, column, MPI_COMM_WORLD);
                    }
                    break;
                }
            }
        }
        double t = (MPI_Wtime() - t0) / reps / 2;          // one-way time per message
        if (rank == 0)
            printf(" %16.2f", 1e6 * t);
    }
    if (rank == 0)
        printf("\n");
    MPI_Type_free(&column);
}

free(a);
free(buf);
free(packbuf);

In [ ]:
# plot the table printed by the benchmark above (requires matplotlib)
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FuncFormatter

surface, ink, muted, grid, axis = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df', '#a3a29c'
colors = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']

lines = [line for line in colbench.splitlines() if line.strip()]
header_line = next(l for l in lines if l.split()[0] == 'n')
# the method names contain blanks, so split the header by the fixed column width
names = [header_line[8 + 17 * k:8 + 17 * (k + 1)].strip() for k in range((len(header_line) - 8) // 17)]
rows = [l.split() for l in lines if l.split()[0].isdigit()]
n = [int(r[0]) for r in rows]

fig, ax = plt.subplots(figsize=(9, 4.8), facecolor=surface)
ax.set_facecolor(surface)
for k, name in enumerate(names):
    ax.plot(n, [float(r[k + 1]) for r in rows], color=colors[k], linewidth=2, marker='o', markersize=5, label=name)
ax.set_xscale('log')
ax.set_yscale('log')
ax.xaxis.set_major_locator(FixedLocator(n))
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
ax.xaxis.set_minor_locator(FixedLocator([]))
ax.set_title('Time per message [µs] — one row or column of an n × n matrix of doubles', loc='left', fontsize=12, color=ink)
ax.set_xlabel('matrix dimension n', color=muted)
ax.grid(True, which='major', color=grid, linewidth=0.8)
ax.tick_params(colors=muted, which='both')
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
for side in ('left', 'bottom'):
    ax.spines[side].set_color(axis)
ax.legend(loc='upper left', frameon=False, labelcolor=ink)
fig.tight_layout()
plt.show()

Questions:
* How much slower than the row is the column with the derived type? Where does the difference come from — the communication, or the memory access?
* Is the derived type faster or slower than manual packing in your MPI library?
* Why is the element-wise version so slow? Compare its time with `n` times the latency of a single small message — the time of the row for small `n` is a good estimate.

## Exercises

### Exercise 1: Upper Triangle and Diagonal

1. Create an indexed type for the **upper triangle** of an `N × N` matrix, including the diagonal, and use the "receive as contiguous" trick to check it.
2. Create a type for the **diagonal** with `MPI_Type_create_indexed_block()`, which takes a single block length for all blocks:

```c
int MPI_Type_create_indexed_block(int count, int blocklength, const int displacements[],
                                  MPI_Datatype oldtype, MPI_Datatype *newtype);
```

Which other constructor could describe the diagonal?

<details>
<summary>Solution</summary>

```c
int lens[N], displs[N];
for (int i = 0; i < N; ++i) {
    lens[i]   = N - i;                          // row i from column i to the end
    displs[i] = i * N + i;
}
MPI_Datatype triu, diag;
MPI_Type_indexed(N, lens, displs, MPI_INT, &triu);
MPI_Type_commit(&triu);
MPI_Sendrecv(a, 1, triu, rank, 0, b, N * (N + 1) / 2, MPI_INT, rank, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
printf("upper triangle:");
for (int i = 0; i < N * (N + 1) / 2; ++i)
    printf(" %d", b[i]);
printf("\n");

for (int i = 0; i < N; ++i)
    displs[i] = i * (N + 1);
MPI_Type_create_indexed_block(N, 1, displs, MPI_INT, &diag);
MPI_Type_commit(&diag);
MPI_Sendrecv(a, 1, diag, rank, 0, b, N, MPI_INT, rank, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
printf("diagonal:");
for (int i = 0; i < N; ++i)
    printf(" %d", b[i]);
printf("\n");

MPI_Type_free(&triu);
MPI_Type_free(&diag);
```

The diagonal is also a vector: `MPI_Type_vector(N, 1, N + 1, MPI_INT, &diag)`.
</details>

In [ ]:
%%mpi -n 1
#define N 5

int a[N * N], b[N * (N + 1) / 2];
for (int i = 0; i < N * N; ++i)
    a[i] = i;

// TODO 1: the upper triangle including the diagonal, received as contiguous ints into b
// TODO 2: the diagonal with MPI_Type_create_indexed_block()

### Exercise 2: Matrix Transpose with Data Types

Transpose an `NR × NC` matrix `a` into the `NC × NR` matrix `b` — without a single loop over the elements, only with a data type:
send `a` as `NR · NC` contiguous `int`s, and receive it into `b` with a type that places the rows of `a` into the **columns** of `b`.

Hint: a column of `b` is a vector; what must its extent be, so that `NR` copies of it fill the columns of `b` one after the other?

<details>
<summary>Solution</summary>

```c
MPI_Datatype col, col1;
MPI_Type_vector(NC, 1, NR, MPI_INT, &col);              // a column of b: NC elements, stride NR
MPI_Type_create_resized(col, 0, sizeof(int), &col1);    // the next column starts one int later
MPI_Type_commit(&col1);

MPI_Sendrecv(a, NR * NC, MPI_INT, rank, 0, b, NR, col1, rank, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

MPI_Type_free(&col);
MPI_Type_free(&col1);
```

`col` is only a building block and need not be committed.
Resized column types of this kind are also the key to transposing a matrix that is distributed by rows with a single `MPI_Alltoall()`.
</details>

In [ ]:
%%mpi -n 1
#define NR 3
#define NC 4

int a[NR][NC], b[NC][NR] = { { 0 } };
for (int r = 0; r < NR; ++r)
    for (int c = 0; c < NC; ++c)
        a[r][c] = 10 * r + c;

// TODO: receive a into b with a data type, so that b is the transpose of a

for (int r = 0; r < NC; ++r) {
    for (int c = 0; c < NR; ++c)
        printf(" %2d", b[r][c]);
    printf("\n");
}

### Exercise 3: Particles

Rank 0 holds an array of particles, `struct particle { double x[3]; double m; int id; }`.
Create a struct type for a whole particle with `MPI_Get_address()`, check that its extent equals `sizeof(struct particle)`, and send all particles to rank 1 in one message.

<details>
<summary>Solution</summary>

```c
MPI_Datatype ptype;
int          blocklengths[3] = { 3, 1, 1 };
MPI_Datatype types[3]        = { MPI_DOUBLE, MPI_DOUBLE, MPI_INT };
MPI_Aint     displs[3], base;

MPI_Get_address(&p[0],    &base);
MPI_Get_address(&p[0].x,  &displs[0]);
MPI_Get_address(&p[0].m,  &displs[1]);
MPI_Get_address(&p[0].id, &displs[2]);
for (int i = 0; i < 3; ++i)
    displs[i] = MPI_Aint_diff(displs[i], base);

MPI_Type_create_struct(3, blocklengths, displs, types, &ptype);
MPI_Type_commit(&ptype);

MPI_Aint lb, extent;
MPI_Type_get_extent(ptype, &lb, &extent);
if (rank == 0)
    printf("extent %ld, sizeof %zu\n", (long)extent, sizeof(struct particle));

if (rank == 0) {
    MPI_Send(p, NP, ptype, 1, 0, MPI_COMM_WORLD);
} else if (rank == 1) {
    MPI_Recv(p, NP, ptype, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    for (int k = 0; k < NP; ++k)
        printf("rank 1: particle %d at (%g, %g, %g), mass %g\n", p[k].id, p[k].x[0], p[k].x[1], p[k].x[2], p[k].m);
}

MPI_Type_free(&ptype);
```

If the extent differed from `sizeof` — e.g., with unusual compiler settings for the alignment — `MPI_Type_create_resized(ptype, 0, sizeof(struct particle), …)` would fix it.
</details>

In [ ]:
%%mpi -n 2
struct particle { double x[3]; double m; int id; };
#define NP 4

struct particle p[NP];
if (rank == 0) {
    for (int k = 0; k < NP; ++k) {
        for (int d = 0; d < 3; ++d)
            p[k].x[d] = k + 0.1 * d;
        p[k].m  = 1.0 + k;
        p[k].id = 100 + k;
    }
}

// TODO: create a type for one particle, check its extent, and send all particles from rank 0 to rank 1

### Exercise 4: A Sparse Vector

Rank 0 holds a vector `v` of `N` `double`s, most of which are zero.
Send only the nonzeros to rank 1, in **one** message: pack the number of nonzeros `nnz`, then their `nnz` indices, then their `nnz` values.
Rank 1 does not know `nnz` in advance: probe the message, unpack it, reconstruct the full vector, and print it.

How many bytes does the message have, compared with sending the whole vector?
From which fraction of nonzeros on is it cheaper to send the whole vector?

<details>
<summary>Solution</summary>

```c
if (rank == 0) {
    // collect the indices and values of the nonzeros
    int nnz = 0, idx[N];
    double val[N];
    for (int i = 0; i < N; ++i)
        if (v[i] != 0.0) {
            idx[nnz] = i;
            val[nnz] = v[i];
            ++nnz;
        }

    // one bound for each of the three MPI_Pack() calls
    int s1, s2, s3;
    MPI_Pack_size(1,   MPI_INT,    MPI_COMM_WORLD, &s1);
    MPI_Pack_size(nnz, MPI_INT,    MPI_COMM_WORLD, &s2);
    MPI_Pack_size(nnz, MPI_DOUBLE, MPI_COMM_WORLD, &s3);
    const int bufsize = s1 + s2 + s3;
    char *buffer = malloc(bufsize);

    int position = 0;
    MPI_Pack(&nnz, 1,   MPI_INT,    buffer, bufsize, &position, MPI_COMM_WORLD);
    MPI_Pack(idx,  nnz, MPI_INT,    buffer, bufsize, &position, MPI_COMM_WORLD);
    MPI_Pack(val,  nnz, MPI_DOUBLE, buffer, bufsize, &position, MPI_COMM_WORLD);

    printf("rank 0: %d nonzeros in %d bytes instead of %zu\n", nnz, position, sizeof v);
    MPI_Send(buffer, position, MPI_PACKED, 1, 0, MPI_COMM_WORLD);
    free(buffer);
} else if (rank == 1) {
    MPI_Status status;
    int nbytes;
    MPI_Probe(0, 0, MPI_COMM_WORLD, &status);
    MPI_Get_count(&status, MPI_PACKED, &nbytes);
    char *buffer = malloc(nbytes);
    MPI_Recv(buffer, nbytes, MPI_PACKED, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

    int position = 0, nnz;
    MPI_Unpack(buffer, nbytes, &position, &nnz, 1, MPI_INT, MPI_COMM_WORLD);
    int    *idx = malloc(nnz * sizeof(int));
    double *val = malloc(nnz * sizeof(double));
    MPI_Unpack(buffer, nbytes, &position, idx, nnz, MPI_INT,    MPI_COMM_WORLD);
    MPI_Unpack(buffer, nbytes, &position, val, nnz, MPI_DOUBLE, MPI_COMM_WORLD);
    for (int k = 0; k < nnz; ++k)
        v[idx[k]] = val[k];

    printf("rank 1:");
    for (int i = 0; i < N; ++i)
        printf(" %g", v[i]);
    printf("\n");
    free(idx);
    free(val);
    free(buffer);
}
```

The message has 4 + 4 · 4 + 4 · 8 = 52 bytes instead of 160.
Every nonzero costs 12 bytes instead of 8, so the sparse format pays off as long as fewer than about two thirds of the entries are nonzero — in terms of message size; packing costs time, too.

Packing every nonzero with its own `MPI_Pack()` call would also work, but then the buffer must hold `nnz` times the bound for a single element: `MPI_Pack_size()` bounds one call, not one element.

The copy into `val` can be avoided with a derived type: the sender creates `MPI_Type_create_indexed_block(nnz, 1, idx, MPI_DOUBLE, &nztype)` and packs `v` with it, and the receiver unpacks the values as `nnz` contiguous `double`s.
Only the sender can build this type — the receiver learns `idx` from the message itself.
</details>

In [ ]:
%%mpi -n 2
#define N 20

double v[N] = { 0.0 };
if (rank == 0) {
    v[2] = 1.5; v[3] = -2.0; v[11] = 4.25; v[17] = 0.5;
}

// TODO: rank 0 packs the number of nonzeros, their indices and their values into one message;
//       rank 1 probes the message, unpacks it into v, and prints the full vector

## Summary

* A data type is described by its **type map**; only the **type signatures** of sender and receiver must match.
* Derived types are constructed, committed with `MPI_Type_commit()` and freed with `MPI_Type_free()` — all local calls.
* The **size** is the amount of data, the **extent** determines where the next copy of a type starts; `MPI_Type_create_resized()` changes the extent.
* `MPI_Type_create_subarray()` selects a block of a multi-dimensional array; the buffer is the start of the full array.
* `MPI_Pack()` and `MPI_Unpack()` gather and scatter data explicitly through a buffer that is sent as `MPI_PACKED` — for messages whose layout is only known at run time; `position` is the cursor.

| Call | Purpose |
|---|---|
| `MPI_Type_contiguous()` | `count` consecutive copies of a type |
| `MPI_Type_vector()`, `MPI_Type_create_hvector()` | equally spaced blocks, e.g. a column (stride in elements / bytes) |
| `MPI_Type_indexed()`, `MPI_Type_create_indexed_block()`, `MPI_Type_create_hindexed()` | blocks at arbitrary positions |
| `MPI_Type_create_struct()` | blocks of different types, e.g. a C struct |
| `MPI_Type_create_subarray()` | a rectangular block of a multi-dimensional array |
| `MPI_Type_create_resized()` | change lower bound and extent |
| `MPI_Type_commit()`, `MPI_Type_free()` | make a type usable / release it |
| `MPI_Type_size()`, `MPI_Type_get_extent()`, `MPI_Type_get_true_extent()` | query a type |
| `MPI_Get_address()`, `MPI_Aint_diff()`, `MPI_Aint_add()` | portable address arithmetic |
| `MPI_Pack()`, `MPI_Unpack()`, `MPI_Pack_size()` | explicit packing |
| `MPI_Pack_external()`, `MPI_Unpack_external()`, `MPI_Pack_external_size()` | portable packing in the `external32` representation |

Next: [Virtual Topologies and Neighborhood Collectives](topologies.ipynb) puts row and column types to work in a 2D ghost cell exchange.

[Back to the overview](../overview.ipynb)